<a href="https://colab.research.google.com/github/ChinthakindhiNishitha/DeepLearining/blob/main/week7%2C8(LeNet%2CAlexNet_).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import h5py
from keras.models import Sequential
from keras.layers import Conv2D, AveragePooling2D, Flatten, Dense, Dropout, Input
from keras.utils import to_categorical
from sklearn.model_selection import train_test_split

# ================================
# LOAD DATASET (CORRECT KEYS)
# ================================
train_path = "/content/drive/MyDrive/Signs_Data_Training.h5"
test_path = "/content/drive/MyDrive/Signs_Data_Testing.h5"

# Load training data
with h5py.File(train_path, "r") as f:
    print("Train keys:", list(f.keys()))
    x_train = np.array(f["train_set_x"])
    y_train = np.array(f["train_set_y"])

# Load testing data
with h5py.File(test_path, "r") as f:
    print("Test keys:", list(f.keys()))
    x_test = np.array(f["test_set_x"])
    y_test = np.array(f["test_set_y"])

print("Train shape:", x_train.shape)
print("Test shape:", x_test.shape)

# ================================
# PREPROCESSING
# ================================

# Normalize
x_train = x_train / 255.0
x_test = x_test / 255.0

# Train-validation split
x_train_main, x_val, y_train_main, y_val = train_test_split(
    x_train, y_train, test_size=0.2, random_state=42
)

# One-hot encoding
num_classes = len(np.unique(y_train))

y_train_main = to_categorical(y_train_main, num_classes)
y_val = to_categorical(y_val, num_classes)
y_test = to_categorical(y_test, num_classes)

# ================================
# BUILD LeNet-5 MODEL
# ================================
input_shape = x_train.shape[1:]  # e.g., (64, 64, 3)

model = Sequential()

# Keras recommends using Input(shape) as the first layer in Sequential models
model.add(Input(shape=input_shape))
model.add(Conv2D(6, (5, 5), activation='tanh'))
model.add(AveragePooling2D((2, 2)))

model.add(Conv2D(16, (5, 5), activation='tanh'))
model.add(AveragePooling2D((2, 2)))

model.add(Flatten())

model.add(Dense(120, activation='tanh'))
model.add(Dropout(0.2)) # Added Dropout layer
model.add(Dense(84, activation='tanh'))
model.add(Dense(num_classes, activation='softmax'))

# ================================
# COMPILE & TRAIN
# ================================
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

model.summary()

model.fit(
    x_train_main, y_train_main,
    epochs=20,
    batch_size=128,
    validation_data=(x_val, y_val)
)

# ================================
# EVALUATE
# ================================
test_loss, test_acc = model.evaluate(x_test, y_test)
print("Test Accuracy:", test_acc)

In [ ]:
# ================================
# IMPORT LIBRARIES
# ================================
import numpy as np
import h5py
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.utils import to_categorical

# ================================
# LOAD DATASET
# ================================
def load_dataset():
    with h5py.File("/content/drive/MyDrive/Signs_Data_Training.h5", "r") as f:
        X_train = np.array(f["train_set_x"])
        Y_train = np.array(f["train_set_y"])

    with h5py.File("/content/drive/MyDrive/Signs_Data_Testing.h5", "r") as f:
        X_test = np.array(f["test_set_x"])
        Y_test = np.array(f["test_set_y"])

    # Normalize
    X_train = X_train / 255.0
    X_test = X_test / 255.0

    # One-hot encoding
    Y_train = to_categorical(Y_train, 6)
    Y_test = to_categorical(Y_test, 6)

    return X_train, Y_train, X_test, Y_test


# ================================
# TUNED LeNet-5 MODEL
# ================================
def build_tuned_lenet():

    model = models.Sequential([

        layers.Input(shape=(64,64,3)),

        # ?? Conv Block 1
        layers.Conv2D(32, (5,5), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2,2)),

        # ?? Conv Block 2
        layers.Conv2D(64, (5,5), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2,2)),

        # ?? Conv Block 3 (NEW - added depth)
        layers.Conv2D(128, (3,3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2,2)),

        layers.Flatten(),

        # ?? Fully Connected Layers
        layers.Dense(120, activation='relu'),


        layers.Dense(84, activation='relu'),


        layers.Dense(6, activation='softmax')
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    return model


# ================================
# TRAIN & EVALUATE
# ================================
X_train, Y_train, X_test, Y_test = load_dataset()

model = build_tuned_lenet()

model.summary()

history = model.fit(
    X_train,
    Y_train,
    epochs=20,
    batch_size=64,
    validation_split=0.2
)

test_loss, test_acc = model.evaluate(X_test, Y_test)
print("Test Accuracy:", test_acc)

In [ ]:
import tensorflow as tf
from tensorflow import keras
from keras import Sequential
from keras.layers import Conv2D, MaxPooling2D, Dense, Flatten, Dropout, BatchNormalization
import numpy as np
import h5py
import matplotlib.pyplot as plt
import gc

# Clear session
keras.backend.clear_session()

# -------------------------------
# 1. Load Dataset
# -------------------------------
def load_dataset():
    with h5py.File("/content/drive/MyDrive/Signs_Data_Training.h5", "r") as f:
        X_train = np.array(f["train_set_x"])
        Y_train = np.array(f["train_set_y"])

    with h5py.File("/content/drive/MyDrive/Signs_Data_Testing.h5", "r") as f:
        X_test = np.array(f["test_set_x"])
        Y_test = np.array(f["test_set_y"])

    return X_train, Y_train, X_test, Y_test

X_train, Y_train, X_test, Y_test = load_dataset()

print("Train shape:", X_train.shape)

# -------------------------------
# 2. Preprocessing
# -------------------------------

# Normalize
X_train = X_train / 255.0
X_test = X_test / 255.0

# Classes
num_classes = len(np.unique(Y_train))

# One-hot encoding
Y_train = keras.utils.to_categorical(Y_train, num_classes)
Y_test = keras.utils.to_categorical(Y_test, num_classes)

# Resize to AlexNet size (227x227 is original, using 224 or 112 is ok)
IMG_SIZE = 112
X_train = tf.image.resize(X_train, (IMG_SIZE, IMG_SIZE))
X_test = tf.image.resize(X_test, (IMG_SIZE, IMG_SIZE))

# -------------------------------
# 3. Dataset Pipeline
# -------------------------------
train_dataset = tf.data.Dataset.from_tensor_slices((X_train, Y_train))
train_dataset = train_dataset.shuffle(1000).batch(8).prefetch(tf.data.AUTOTUNE)

test_dataset = tf.data.Dataset.from_tensor_slices((X_test, Y_test))
test_dataset = test_dataset.batch(8).prefetch(tf.data.AUTOTUNE)

# -------------------------------
# 4. AlexNet Model
# -------------------------------
model = Sequential()

# Layer 1
model.add(Conv2D(96, (11,11), strides=4, activation='relu', input_shape=X_train.shape[1:]))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(3,3), strides=2))

# Layer 2
model.add(Conv2D(256, (5,5), padding='same', activation='relu'))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(3,3), strides=2))

# Layer 3
model.add(Conv2D(384, (3,3), padding='same', activation='relu'))

# Layer 4
model.add(Conv2D(384, (3,3), padding='same', activation='relu'))

# Layer 5
model.add(Conv2D(256, (3,3), padding='same', activation='relu'))
model.add(MaxPooling2D(pool_size=(3,3), strides=2))

# Fully Connected Layers
model.add(Flatten())

model.add(Dense(4096, activation='relu'))
model.add(Dropout(0.5))

model.add(Dense(4096, activation='relu'))
model.add(Dropout(0.5))

# Output
model.add(Dense(num_classes, activation='softmax'))

# -------------------------------
# 5. Compile
# -------------------------------
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.0001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

# -------------------------------
# 6. Train
# -------------------------------
history = model.fit(
    train_dataset,
    epochs=10,
    validation_data=test_dataset
)

# -------------------------------
# 7. Evaluate
# -------------------------------
test_loss, test_acc = model.evaluate(test_dataset)
print("Test Accuracy:", test_acc)

# -------------------------------
# 8. Plot
# -------------------------------
plt.figure(figsize=(12,4))

plt.subplot(1,2,1)
plt.plot(history.history['accuracy'], label='Train')
plt.plot(history.history['val_accuracy'], label='Validation')
plt.legend()
plt.title('Accuracy')

plt.subplot(1,2,2)
plt.plot(history.history['loss'], label='Train')
plt.plot(history.history['val_loss'], label='Validation')
plt.legend()
plt.title('Loss')

plt.show()

# -------------------------------
# 9. Clear memory
# -------------------------------
gc.collect()

In [ ]:
import tensorflow as tf
from tensorflow import keras
from keras import Sequential
from keras.layers import Conv2D, MaxPooling2D, Dense, Flatten, Dropout, BatchNormalization
import numpy as np
import h5py
import matplotlib.pyplot as plt
import gc

# Clear session
keras.backend.clear_session()

# -------------------------------
# 1. Load Dataset
# -------------------------------
def load_dataset():
    with h5py.File("/content/drive/MyDrive/Signs_Data_Training.h5", "r") as f:
        X_train = np.array(f["train_set_x"])
        Y_train = np.array(f["train_set_y"])

    with h5py.File("/content/drive/MyDrive/Signs_Data_Testing.h5", "r") as f:
        X_test = np.array(f["test_set_x"])
        Y_test = np.array(f["test_set_y"])

    return X_train, Y_train, X_test, Y_test

X_train, Y_train, X_test, Y_test = load_dataset()

print("Train shape:", X_train.shape)

# -------------------------------
# 2. Preprocessing
# -------------------------------

# Normalize
X_train = X_train / 255.0
X_test = X_test / 255.0

# Classes
num_classes = len(np.unique(Y_train))

# One-hot encoding
Y_train = keras.utils.to_categorical(Y_train, num_classes)
Y_test = keras.utils.to_categorical(Y_test, num_classes)

# Resize (ZFNet uses ~224, but 112 saves memory)
IMG_SIZE = 112
X_train = tf.image.resize(X_train, (IMG_SIZE, IMG_SIZE))
X_test = tf.image.resize(X_test, (IMG_SIZE, IMG_SIZE))

# -------------------------------
# 3. Dataset Pipeline
# -------------------------------
train_dataset = tf.data.Dataset.from_tensor_slices((X_train, Y_train))
train_dataset = train_dataset.shuffle(1000).batch(8).prefetch(tf.data.AUTOTUNE)

test_dataset = tf.data.Dataset.from_tensor_slices((X_test, Y_test))
test_dataset = test_dataset.batch(8).prefetch(tf.data.AUTOTUNE)

# -------------------------------
# 4. ZFNet Model
# -------------------------------
model = Sequential()

# Layer 1 (KEY CHANGE: 7x7, stride 2)
model.add(Conv2D(96, (7,7), strides=2, activation='relu', input_shape=X_train.shape[1:]))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(3,3), strides=2))

# Layer 2
model.add(Conv2D(256, (5,5), padding='same', activation='relu'))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(3,3), strides=2))

# Layer 3
model.add(Conv2D(384, (3,3), padding='same', activation='relu'))

# Layer 4
model.add(Conv2D(384, (3,3), padding='same', activation='relu'))

# Layer 5
model.add(Conv2D(256, (3,3), padding='same', activation='relu'))
model.add(MaxPooling2D(pool_size=(3,3), strides=2))

# Fully Connected Layers
model.add(Flatten())

model.add(Dense(4096, activation='relu'))
model.add(Dropout(0.5))

model.add(Dense(4096, activation='relu'))
model.add(Dropout(0.5))

# Output
model.add(Dense(num_classes, activation='softmax'))

# -------------------------------
# 5. Compile
# -------------------------------
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.0001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

# -------------------------------
# 6. Train
# -------------------------------
history = model.fit(
    train_dataset,
    epochs=10,
    validation_data=test_dataset
)

# -------------------------------
# 7. Evaluate
# -------------------------------
test_loss, test_acc = model.evaluate(test_dataset)
print("Test Accuracy:", test_acc)

# -------------------------------
# 8. Plot
# -------------------------------
plt.figure(figsize=(12,4))

plt.subplot(1,2,1)
plt.plot(history.history['accuracy'], label='Train')
plt.plot(history.history['val_accuracy'], label='Validation')
plt.legend()
plt.title('Accuracy')

plt.subplot(1,2,2)
plt.plot(history.history['loss'], label='Train')
plt.plot(history.history['val_loss'], label='Validation')
plt.legend()
plt.title('Loss')

plt.show()

# -------------------------------
# 9. Clear memory
# -------------------------------
gc.collect()

VGG

In [ ]:
import tensorflow as tf
from tensorflow import keras
from keras.models import Model
from keras.layers import Dense, Flatten, Dropout
import numpy as np
import h5py
import matplotlib.pyplot as plt
import gc

# Clear session
keras.backend.clear_session()

# -------------------------------
# 1. Load Dataset
# -------------------------------
def load_dataset():
    with h5py.File("/content/drive/MyDrive/Signs_Data_Training.h5", "r") as f:
        X_train = np.array(f["train_set_x"])
        Y_train = np.array(f["train_set_y"])

    with h5py.File("/content/drive/MyDrive/Signs_Data_Testing.h5", "r") as f:
        X_test = np.array(f["test_set_x"])
        Y_test = np.array(f["test_set_y"])

    return X_train, Y_train, X_test, Y_test

X_train, Y_train, X_test, Y_test = load_dataset()

print("Train shape:", X_train.shape)

# -------------------------------
# 2. Preprocessing
# -------------------------------

# Normalize
X_train = X_train / 255.0
X_test = X_test / 255.0

# Classes
num_classes = len(np.unique(Y_train))

# One-hot encoding
Y_train = keras.utils.to_categorical(Y_train, num_classes)
Y_test = keras.utils.to_categorical(Y_test, num_classes)

# Resize to VGG size (224x224)
IMG_SIZE = 224
X_train = tf.image.resize(X_train, (IMG_SIZE, IMG_SIZE))
X_test = tf.image.resize(X_test, (IMG_SIZE, IMG_SIZE))

# -------------------------------
# 3. Load VGG16 Base Model
# -------------------------------
base_model = tf.keras.applications.VGG16(
    weights='imagenet',
    include_top=False,
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)

# Freeze base layers
for layer in base_model.layers:
    layer.trainable = False

# -------------------------------
# 4. Add Custom Layers
# -------------------------------
x = base_model.output
x = Flatten()(x)
x = Dense(256, activation='relu')(x)
x = Dropout(0.5)(x)

output = Dense(num_classes, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=output)

# -------------------------------
# 5. Compile
# -------------------------------
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

# -------------------------------
# 6. Train
# -------------------------------
history = model.fit(
    X_train, Y_train,
    epochs=10,
    batch_size=16,
    validation_data=(X_test, Y_test)
)

# -------------------------------
# 7. Evaluate
# -------------------------------
test_loss, test_acc = model.evaluate(X_test, Y_test)
print("Test Accuracy:", test_acc)

# -------------------------------
# 8. Plot
# -------------------------------
plt.figure(figsize=(12,4))

plt.subplot(1,2,1)
plt.plot(history.history['accuracy'], label='Train')
plt.plot(history.history['val_accuracy'], label='Validation')
plt.legend()
plt.title('Accuracy')

plt.subplot(1,2,2)
plt.plot(history.history['loss'], label='Train')
plt.plot(history.history['val_loss'], label='Validation')
plt.legend()
plt.title('Loss')

plt.show()

# -------------------------------
# 9. Clear memory
# -------------------------------
gc.collect()

**GoogleNet**

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models, transforms
from torch.utils.data import Dataset, DataLoader
import numpy as np
import h5py
from PIL import Image

# -------------------------------
# 1. Load Dataset
# -------------------------------
def load_dataset():
    with h5py.File("/content/drive/MyDrive/Signs_Data_Training.h5", "r") as f:
        X_train = np.array(f["train_set_x"])
        Y_train = np.array(f["train_set_y"])

    with h5py.File("/content/drive/MyDrive/Signs_Data_Testing.h5", "r") as f:
        X_test = np.array(f["test_set_x"])
        Y_test = np.array(f["test_set_y"])

    return X_train, Y_train, X_test, Y_test

X_train, Y_train, X_test, Y_test = load_dataset()

num_classes = len(np.unique(Y_train))

# -------------------------------
# 2. Custom Dataset Class
# -------------------------------
class SignDataset(Dataset):
    def __init__(self, X, Y, transform=None):
        self.X = X
        self.Y = Y
        self.transform = transform

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        image = self.X[idx]
        label = self.Y[idx]

        image = Image.fromarray(image.astype('uint8'))

        if self.transform:
            image = self.transform(image)

        return image, label

# -------------------------------
# 3. Transforms (IMPORTANT)
# -------------------------------
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# -------------------------------
# 4. DataLoaders
# -------------------------------
train_dataset = SignDataset(X_train, Y_train, transform)
test_dataset = SignDataset(X_test, Y_test, transform)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16)

# -------------------------------
# 5. Load GoogLeNet Model
# -------------------------------
model = models.googlenet(pretrained=True)

# Replace final layer
model.fc = nn.Linear(model.fc.in_features, num_classes)

# -------------------------------
# 6. Device (GPU/CPU)
# -------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# -------------------------------
# 7. Loss & Optimizer
# -------------------------------
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)

# -------------------------------
# 8. Training Loop
# -------------------------------
epochs = 5

for epoch in range(epochs):
    model.train()
    running_loss = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)

        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {running_loss:.4f}")

# -------------------------------
# 9. Evaluation
# -------------------------------
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print("Test Accuracy:", correct / total)

**ResNet**

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models, transforms
from torch.utils.data import Dataset, DataLoader
import numpy as np
import h5py
from PIL import Image

# -------------------------------
# 1. Load Dataset
# -------------------------------
def load_dataset():
    with h5py.File("/content/drive/MyDrive/Signs_Data_Training.h5", "r") as f:
        X_train = np.array(f["train_set_x"])
        Y_train = np.array(f["train_set_y"])

    with h5py.File("/content/drive/MyDrive/Signs_Data_Testing.h5", "r") as f:
        X_test = np.array(f["test_set_x"])
        Y_test = np.array(f["test_set_y"])

    return X_train, Y_train, X_test, Y_test

X_train, Y_train, X_test, Y_test = load_dataset()

num_classes = len(np.unique(Y_train))

# -------------------------------
# 2. Custom Dataset
# -------------------------------
class SignDataset(Dataset):
    def __init__(self, X, Y, transform=None):
        self.X = X
        self.Y = Y
        self.transform = transform

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        image = self.X[idx]
        label = self.Y[idx]

        image = Image.fromarray(image.astype('uint8'))

        if self.transform:
            image = self.transform(image)

        return image, label

# -------------------------------
# 3. Transforms
# -------------------------------
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# -------------------------------
# 4. DataLoaders
# -------------------------------
train_dataset = SignDataset(X_train, Y_train, transform)
test_dataset = SignDataset(X_test, Y_test, transform)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16)

# -------------------------------
# 5. Load ResNet Model
# -------------------------------
model = models.resnet18(pretrained=True)   # you can try resnet34, resnet50

# Replace final layer
model.fc = nn.Linear(model.fc.in_features, num_classes)

# -------------------------------
# 6. Device
# -------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# -------------------------------
# 7. Loss & Optimizer
# -------------------------------
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)

# -------------------------------
# 8. Training
# -------------------------------
epochs = 5

for epoch in range(epochs):
    model.train()
    running_loss = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)

        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {running_loss:.4f}")

# -------------------------------
# 9. Evaluation
# -------------------------------
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print("Test Accuracy:", correct / total)

?? Accuracy Comparison
Model	Params	Accuracy
LeNet	0.33M	81.6%
LeNet + Dropout	1.12M	? 33.3%
AlexNet	24.7M	63.3%
ZFNet	46.7M	90.8%
VGG	21.1M	97.5%
GoogLeNet	?	99.16% ??
ResNet	?	98.33% ??
?? Key Observations
1. Best Model ?

?? GoogLeNet performs best (99.16%)

Efficient architecture
Inception modules capture multi-scale features
2. Second Best ??

?? ResNet (98.33%)

Residual connections improve learning
Very stable training
3. VGG Performance ??

?? VGG (97.5%)

Deep + small filters
High accuracy but heavy
4. ZFNet Insight ??

?? ZFNet (90.8%) > AlexNet (63.3%)

Shows improvement due to:
smaller filters
smaller stride
5. AlexNet Issue ??

?? Low accuracy (63%)

Too coarse (large filters)
Not suitable for your dataset
6. LeNet Behavior

?? Basic model ? decent accuracy (81%)

Works but not powerful enough
7. Dropout Problem ??

?? LeNet + Dropout ? very low accuracy (33%)

Too much dropout ? underfitting (high bias)